# Figure 3c — AF3 (31 - PTI-PAE) vs TCR:antigen affinity (Kd)

Runnable panel. Reads `fig3c_data.csv` (one row per triad: group, mhc_class, Kd_M,
pti_pae_raw) and plots **31 - PTI-PAE** vs measured Kd, consistent with the rest of
the package.

- 67 class I + 4 class II cognate triads with measured Kd (matches the manuscript).
- 8 engineered sub-uM TCRs (blue), 63 natural TCRs (green).
- 31,590 matched non-cognate controls (grey box, left).
- Spearman (31 - PTI-PAE vs Kd): all cognate r = -0.40 (p = 4.7e-4);
  natural >1 uM r = -0.26 (p = 0.044). Negative because 31 - PTI-PAE inverts the
  raw interface PAE, so a more confident interface goes with tighter (lower) Kd.

`corr_spr_upstream.ipynb` is Lawson's original pipeline notebook (needs the full
repo + git-LFS data); this notebook reproduces its Fig 3c panel from the extracted CSV.


In [1]:
import pandas as pd, numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats

df = pd.read_csv('fig3c_data.csv')
df['inv'] = 31 - df['pti_pae_raw']
cog = df[df['group']=='cognate'].copy(); non = df[df['group']=='noncognate'].copy()
cog_kd, cog_inv, non_inv = cog['Kd_M'].to_numpy(), cog['inv'].to_numpy(), non['inv'].to_numpy()
GREEN="#2c9c7a"; BLUE="#3a78b8"; RED="#c83737"; BOX="0.55"
nat = cog_kd > 1e-6; eng = ~nat

fig, (ax_non, ax_cog) = plt.subplots(
    1, 2, sharey=True, figsize=(7.0, 5.6),
    gridspec_kw={"width_ratios": [1, 5], "wspace": 0.12})

# --- left: noncognate box ---
ax_non.boxplot([non_inv], positions=[0], widths=0.55, patch_artist=True,
               boxprops=dict(facecolor='none', edgecolor=RED, linewidth=1.6),
               medianprops=dict(color=RED, linewidth=1.6),
               whiskerprops=dict(color=RED, linewidth=1.2),
               capprops=dict(color=RED, linewidth=1.2))
ax_non.set_xticks([0])
ax_non.set_xticklabels([f"Noncognate\n(n={len(non_inv)})"], fontsize=12)
ax_non.set_xlim(-0.7, 0.7)
ax_non.set_ylabel("AF3 PTI-PAE", fontsize=15)
ax_non.spines[['right','top']].set_visible(False)

# --- right: Kd scatter (log, inverted x) ---
ax_cog.set_xscale("log"); ax_cog.invert_xaxis()
lo, hi = int(np.floor(np.log10(cog_kd.min()))), int(np.ceil(np.log10(cog_kd.max())))
ax_cog.set_xlim(10**hi, 10**lo)

# per-decade summary boxes, drawn directly on the log axis so they align
logkd = np.log10(cog_kd)
HW = 0.46  # half-width in log10 units => boxes span (almost) the full decade
for e in range(lo, hi):
    m = (logkd >= e) & (logkd < e+1)
    if m.sum() >= 2:
        v = cog_inv[m]; q1, med, q3 = np.percentile(v, [25, 50, 75])
        vmin, vmax = v.min(), v.max()
        # green if natural-majority decade, blue if engineered-majority
        n_nat = int((cog_kd[m] > 1e-6).sum()); bc = GREEN if n_nat >= (m.sum()-n_nat) else BLUE
        cx = 10**(e+0.5); xl = 10**(e+0.5-HW); xr = 10**(e+0.5+HW)
        ax_cog.plot([xl,xr,xr,xl,xl],[q1,q1,q3,q3,q1], color=bc, lw=1.5, zorder=2)
        ax_cog.plot([xl,xr],[med,med], color=bc, lw=2.0, zorder=2)
        ax_cog.plot([cx,cx],[q3,vmax], color=bc, lw=1.0, zorder=1)
        ax_cog.plot([cx,cx],[q1,vmin], color=bc, lw=1.0, zorder=1)
        ax_cog.plot([xl*1.02,xr*0.98],[vmax,vmax], color=bc, lw=1.0, zorder=1)
        ax_cog.plot([xl*1.02,xr*0.98],[vmin,vmin], color=bc, lw=1.0, zorder=1)

ax_cog.scatter(cog_kd[nat], cog_inv[nat], s=26, c=GREEN, edgecolors='k',
               linewidths=0.4, label=f"natural (n={int(nat.sum())})", zorder=5)
ax_cog.scatter(cog_kd[eng], cog_inv[eng], s=26, c=BLUE, edgecolors='k',
               linewidths=0.4, label=f"engineered sub-uM (n={int(eng.sum())})", zorder=5)
ax_cog.set_xlabel(r"$K_d$ (M)", fontsize=15)
ax_cog.spines[['left','top']].set_visible(False)
ax_cog.tick_params(left=False, labelleft=False)   # hide stray y-ticks between panels

# --- legend + stats OUTSIDE the axes, in the right margin ---
# legend and stats stacked in the empty bottom-right corner of the cognate panel
r_all, p_all = stats.spearmanr(cog_inv, cog_kd)
r_nat, p_nat = stats.spearmanr(cog_inv[nat], cog_kd[nat])
stats_text = (f"Spearman\n"
              f"all cognate:      r = {r_all:+.2f},  p = {p_all:.2g}\n"
              f"natural (>1 \u00b5M): r = {r_nat:+.2f},  p = {p_nat:.2g}")
ax_cog.text(0.99, 0.02, stats_text, transform=ax_cog.transAxes,
            va='bottom', ha='right', fontsize=10, family='monospace',
            bbox=dict(boxstyle="round,pad=0.45", fc="white", ec="0.6", alpha=0.95))
ax_cog.legend(loc='lower right', bbox_to_anchor=(0.99, 0.27),
              fontsize=10, framealpha=0.95, borderaxespad=0,
              handlelength=1.4)

ax_non.set_ylim(-1, 32)
fig.suptitle("AF3 PTI-PAE vs TCR:antigen affinity", fontsize=13, y=0.95)
fig.savefig('Fig3c_affinity_correlation.png', dpi=300, bbox_inches='tight')
print(f"all r={r_all:+.3f} p={p_all:.2g} | natural r={r_nat:+.3f} p={p_nat:.2g} | nat={int(nat.sum())} eng={int(eng.sum())}")


all r=-0.404 p=0.00047 | natural r=-0.255 p=0.044 | nat=63 eng=8
